# makemore Part 1 — scaled 10-character history

Another extension of Karpathy's neural bigram
([makemore Part 1](https://www.youtube.com/watch?v=TCH_1BHY58I)). Earlier we added positional
encoding (loss ~2.317) and 2 characters of history (~2.224). Here we push the history window to
**10 characters** and **scale each character by its age**: the most recent character is left
unscaled (x1.0), and each older character is multiplied by `decay**age`, so its influence fades
the further back it is.

**Input layout (293 dims):** `[ char_t (27) | char_{t-1} (27) | ... | char_{t-9} (27) | position (23) ]`
= `10*27 + 23 = 293`, so `W` is **293 x 27**. The history slots *shift* as the window advances and
are zeroed where they would cross a word boundary. No feedback loop — the input is a fixed function
of the characters, so training is the same single-matmul loop as before.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
print('num words:', len(words))
print('first few:', words[:8])

num words: 32033
first few: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1]); ys.append(stoi[ch2])
xs = torch.tensor(xs); ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

number of examples: 228146


## Position within the word

Same vectorized position counter as the other notebooks: it resets to 0 at every `.` boundary
(`xs == 0`) via a running `cummax`. We use it both for the positional one-hot and to know how many
history slots are valid at each step.

In [5]:
def positions_of(xs):
    idx = torch.arange(xs.shape[0], device=xs.device)
    boundary_idx = torch.where(xs == 0, idx, torch.zeros_like(idx))
    last_boundary = torch.cummax(boundary_idx, dim=0).values
    return idx - last_boundary

demo = torch.tensor([0, 5, 13, 13, 1, 0, 1, 22, 1])   # 'emma' then 'ava'
print('xs:       ', demo.tolist())
print('positions:', positions_of(demo).tolist())

xs:        [0, 5, 13, 13, 1, 0, 1, 22, 1]
positions: [0, 1, 2, 3, 4, 0, 1, 2, 3]


## Encoding: scaled 10-character history + position

For each example we build 10 character one-hots: the current character (slot 0, **unscaled**) and
the 9 previous characters (slots 1..9), each multiplied by `decay**k` for its age `k`. A history
slot is zeroed when it would reach past the start of the current word (`position < k`). The 23-dim
positional one-hot is appended as before.

In [6]:
HIST = 10

def encode_xs(xs, decay=0.9, hist=HIST, max_chars=23):
    N = xs.shape[0]
    pos = positions_of(xs)
    blocks = []
    for k in range(hist):                     # k = 0 current (unscaled), larger k = older
        if k == 0:
            xk = xs
        else:
            xk = torch.zeros(N, dtype=xs.dtype); xk[k:] = xs[:N - k]   # shift right by k
        ek = F.one_hot(xk, num_classes=27).float() * (decay ** k)      # scale by age
        if k > 0:
            ek[pos < k] = 0.0                  # zero history that crosses the word start
        blocks.append(ek)
    blocks.append(F.one_hot(pos.clamp(max=max_chars - 1), num_classes=max_chars).float())
    return torch.cat(blocks, dim=1)           # (N, 10*27 + 23 = 293)

# sanity: for emma's 'a' (position 4) the slots should read [1, .9, .81, .729, .6561, 0, 0, 0, 0, 0]
enc = encode_xs(demo, decay=0.9)
print('encoded dim:', tuple(enc.shape))
print('slot scales at row 4 (emma "a", pos 4):',
      [round(enc[4, k*27:(k+1)*27].max().item(), 4) for k in range(HIST)])

encoded dim: (9, 293)
slot scales at row 4 (emma "a", pos 4): [1.0, 0.9, 0.81, 0.729, 0.6561, 0.0, 0.0, 0.0, 0.0, 0.0]


## Train

Same training loop as the original. The input is wider (293 dims) and each row sums to more than
before, so we use a smaller learning rate (`lr=30`). `decay=0.9` gives each older character a
gently fading vote; higher decay (longer effective memory) helps — see the note below.

In [7]:
DECAY = 0.9
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((293, 27), generator=g, requires_grad=True)

X = encode_xs(xs, decay=DECAY)   # constant across steps -> build once
for k in range(1000):
    logits = X @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    if k % 100 == 0 or k == 999:
        print(k, loss.item())
    W.grad = None
    loss.backward()
    W.data += -30 * W.grad
print('final loss:', loss.item())

0 4.716190338134766


100 2.291797637939453


200 2.242746114730835


300 2.2245240211486816


400 2.2150204181671143


...

600 2.205040216445923


700 2.2020602226257324


800 2.1997768878936768


900 2.1979668140411377


999 2.1965067386627197
final loss: 2.1965067386627197


Progression so far: plain bigram **~2.483** -> + position **~2.317** -> + 2-char history **~2.224**
-> **10-char scaled history ~2.20** (this notebook). A quick `decay` sweep (500 steps, lr=25) shows
longer memory helps monotonically: `0.5 -> 2.239`, `0.7 -> 2.225`, `0.8 -> 2.220`, `0.9 -> 2.214`,
and `0.95` at lr=30/1000 steps reaches **~2.194** — the best single-linear-layer result in this
series. It's still one linear layer; only the input got richer.

## Generate words

Generation keeps the same sliding, age-scaled window: track the emitted character ids, rebuild the
10 scaled slots plus the position one-hot, sample, and shift.

In [8]:
def generate_word(W, generator, decay=DECAY, hist=HIST, max_chars=23):
    out = []; histids = []; ix = 0
    while True:
        pos = min(len(out), max_chars - 1)
        blocks = []
        for k in range(hist):
            if k == 0:
                e = F.one_hot(torch.tensor([ix]), num_classes=27).float()
            elif len(histids) >= k:
                e = F.one_hot(torch.tensor([histids[-k]]), num_classes=27).float() * (decay ** k)
            else:
                e = torch.zeros((1, 27))
            blocks.append(e)
        blocks.append(F.one_hot(torch.tensor([pos]), num_classes=max_chars).float())
        X = torch.cat(blocks, dim=1)                     # (1, 293)
        p = (X @ W).exp(); p = p / p.sum(1, keepdims=True)
        nix = torch.multinomial(p, num_samples=1, replacement=True, generator=generator).item()
        out.append(itos[nix]); histids.append(ix); ix = nix
        if ix == 0:
            break
    return ''.join(out)

def generate(n=20, seed=2147483647, decay=DECAY):
    gg = torch.Generator().manual_seed(seed)
    return [generate_word(W, gg, decay=decay) for _ in range(n)]

for name in generate(20):
    print(name)

cexze.
morlyus.
rochitah.
mellimitta.
nellayn.
kanda.
samiyah.
javer.
gotti.
moriella.
kina.
teda.
kaley.
maside.
enkavi.
nyletls.
migrice.
vorhens.
kasdr.
breell.
